# 02 안전재고 두 식 비교 (S15P31A308-106)

**결론**: (샘플) 위글리 배달 이력 수령 후 작성. 샘플은 하루 판매 변동을 15%로 고르게 만든 가상 데이터라 통계식이 대부분 기존 식의 절반 안팎으로 작게 나온다 — 판정용이 아니라 계산 틀 확인용이다.

- 기존 식: 최근 7일 평균 × N일 (DECISIONS #10) / 통계식: z × σ × √(R+L) (#28, 설계 5.3)
- σ = 최근 28일 영업일 일 소비량의 표본 표준편차, 휴무일 제외 — 백엔드 S15P31A308-108 과 같은 기준
- 튀는 재료 = 통계식 ÷ 기존 식이 2배 초과 또는 0.5배 미만
- 실데이터는 `DATA_DIR` 만 바꿔 다시 돌린다 (같은 CSV 열)


In [ ]:
from pathlib import Path

import pandas as pd

from weglee_ai.safety_stock import compare, daily_consumption

DATA_DIR = Path("..") / "data" / "sample"  # 위글리 실데이터는 ../data/processed
pd.set_option("display.float_format", "{:,.2f}".format)

In [ ]:
def read(name):
    return pd.read_csv(DATA_DIR / f"{name}.csv")
menus, sales = read("menus"), read("pos_sales")
if "menu_id" not in sales:  # 샘플 판매는 메뉴 이름으로 온다
    sales = sales.merge(menus[["menu_id", "name"]], left_on="menu_name", right_on="name")
daily = daily_consumption(sales, read("recipes"))
table = compare(daily, read("ingredients"), read("supplier_products"), read("suppliers"))
print(f"기준일 {daily.index.max().date()} · 영업일 {len(daily)}일")
table[["재료", "단위", "보관", "z", "R", "L", "σ", "CV", "기존 식", "통계식", "비율", "식", "튐", "비고"]]

## 튀는 재료

In [ ]:
table[table["튐"] != ""][["재료", "비율", "튐", "비고"]]